# Día 3 — Notebook Databricks: Blob Storage + PostgreSQL
### Curso: Introducción a Herramientas de Cómputo en la Nube · Azure

---

Este notebook se importa y ejecuta **dentro de Azure Databricks**.

**Instrucciones para importarlo:**
1. En tu workspace de Databricks, ve al panel izquierdo > `Workspace`
2. Haz clic en el menu de tu carpeta > `Import`
3. Selecciona este archivo `.ipynb` y confirma
4. Asegurate de tener un cluster activo antes de ejecutar

**Lo que haremos:**

| Sección | Actividad |
|---------|-----------|
| 1 | Credenciales y configuración |
| 2 | Montar Blob Storage con ABFS |
| 3 | Leer y explorar datos desde Blob con Spark |
| 4 | Conectar a PostgreSQL con JDBC |
| 5 | Leer tabla de PostgreSQL como Spark DataFrame |
| 6 | Escribir resultados de Spark a PostgreSQL |
| 7 | Ejercicio integrador |

> **Nota sobre el cluster:** usa un cluster de tipo `Single Node` con Databricks Runtime 13.x o superior (Python 3.10+). No necesitas configuracion especial de Spark para este curso.

---
## Sección 1 — Credenciales

Rellena estas variables. Son las mismas que usaste en el notebook local.

> En Databricks de produccion las credenciales se guardan en `Secrets` (Databricks Vault). Para el curso usamos variables directas para simplificar.

In [ ]:
# ── BLOB STORAGE ──────────────────────────────────────────────────────────────
STORAGE_ACCOUNT_NAME = "TU_CUENTA"               # nombre del Storage Account
STORAGE_ACCOUNT_KEY  = "TU_CLAVE"                # Access Key del Storage Account
BLOB_CONTAINER       = "mi-contenedor"
BLOB_ARCHIVO_CSV     = "datos.csv"

# ── POSTGRESQL ────────────────────────────────────────────────────────────────
PG_HOST     = "TU-SERVIDOR.postgres.database.azure.com"
PG_PORT     = "5432"
PG_DATABASE = "postgres"
PG_USER     = "TU_USUARIO"
PG_PASSWORD = "TU_CONTRASEÑA"

# URL JDBC para PostgreSQL (Databricks la usa para conectarse)
JDBC_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}?sslmode=require"

print("Credenciales configuradas.")
print(f"  Storage Account : {STORAGE_ACCOUNT_NAME}")
print(f"  JDBC URL        : {JDBC_URL}")

---
## Sección 2 — Conectar Blob Storage con ABFS

ABFS (Azure Blob File System) es el protocolo que usa Spark para leer
archivos de Blob Storage **directamente, sin descargarlos**.

El primer paso es configurar las credenciales del Storage Account
en la sesion de Spark de este cluster.

In [ ]:
# Configurar la clave del Storage Account en Spark
# Esto le dice al cluster como autenticarse con Azure Blob

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

# Ruta ABFS al archivo CSV
# Formato: abfss://<contenedor>@<cuenta>.dfs.core.windows.net/<archivo>
abfs_path = f"abfss://{BLOB_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/{BLOB_ARCHIVO_CSV}"

print("Configuracion de Blob Storage lista.")
print(f"  Ruta ABFS: {abfs_path}")

In [ ]:
# Listar archivos del contenedor desde Spark
contenedor_path = f"abfss://{BLOB_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/"

archivos = dbutils.fs.ls(contenedor_path)

print(f"Archivos en el contenedor '{BLOB_CONTAINER}':")
for f in archivos:
    print(f"  {f.name}  ({f.size / 1024:.1f} KB)")

---
## Sección 3 — Leer y explorar datos desde Blob con Spark

Spark lee el CSV directamente desde Blob y lo convierte en un
**Spark DataFrame**, que puede procesarse de forma distribuida.

In [ ]:
# Leer el CSV desde Blob Storage con Spark
df_spark = (
    spark.read
         .option("header", "true")       # la primera fila tiene nombres de columna
         .option("inferSchema", "true")  # Spark deduce los tipos de dato
         .csv(abfs_path)
)

print(f"Archivo leido desde Blob.")
print(f"  Filas    : {df_spark.count()}")
print(f"  Columnas : {df_spark.columns}")

In [ ]:
# Ver el schema del DataFrame (tipos de dato que Spark infirió)
df_spark.printSchema()

In [ ]:
# Mostrar las primeras filas
# En Databricks, display() genera una tabla interactiva
display(df_spark)

In [ ]:
# Consulta SQL sobre el DataFrame de Spark
# Primero lo registramos como vista temporal
df_spark.createOrReplaceTempView("datos_blob")

# Ahora podemos usar SQL estandar sobre los datos de Blob
resultado = spark.sql("""
    SELECT
        COUNT(*)          AS total_filas,
        COUNT(DISTINCT categoria) AS categorias_unicas
    FROM datos_blob
""")
display(resultado)

In [ ]:
# Agregacion por categoria
# Ajusta el nombre de columna si tu CSV usa nombres diferentes
from pyspark.sql import functions as F

resumen = (
    df_spark
    .groupBy("categoria")
    .agg(
        F.count("*").alias("conteo"),
        F.round(F.avg("valor"), 2).alias("promedio_valor"),
        F.max("valor").alias("max_valor")
    )
    .orderBy(F.desc("conteo"))
)
display(resumen)

---
## Sección 4 — Conectar a PostgreSQL con JDBC

Databricks incluye el driver JDBC de PostgreSQL. Solo necesitamos
la URL de conexion y las credenciales.

> **Antes de ejecutar:** verifica que la IP de salida de tu workspace de Databricks
> este en las reglas del firewall de tu PostgreSQL Flexible Server.
> Puedes encontrar la IP en: `Databricks workspace > Settings > Cloud resources > Network`

In [ ]:
# Propiedades de conexion JDBC
jdbc_props = {
    "user"    : PG_USER,
    "password": PG_PASSWORD,
    "driver"  : "org.postgresql.Driver",
    "sslmode" : "require"
}

print("Propiedades JDBC configuradas.")
print(f"  URL    : {JDBC_URL}")
print(f"  Driver : {jdbc_props['driver']}")

---
## Sección 5 — Leer tabla de PostgreSQL como Spark DataFrame

In [ ]:
# Leer la tabla 'registros' que creamos en el notebook local
df_registros = (
    spark.read
         .jdbc(url=JDBC_URL, table="registros", properties=jdbc_props)
)

print(f"Tabla 'registros' leida desde PostgreSQL.")
print(f"  Filas    : {df_registros.count()}")
print(f"  Columnas : {df_registros.columns}")
display(df_registros)

In [ ]:
# Leer ejecutando una consulta SQL directamente en PostgreSQL
# Util cuando quieres filtrar en origen para no traer toda la tabla

query = "(SELECT * FROM registros WHERE valor > 100 ORDER BY valor DESC LIMIT 20) AS subquery"

df_filtrado = spark.read.jdbc(url=JDBC_URL, table=query, properties=jdbc_props)

print("Registros con valor > 100:")
display(df_filtrado)

In [ ]:
# Leer la tabla de ubicaciones (PostGIS)
# JDBC no entiende el tipo geometry de PostGIS nativo,
# por eso usamos ST_AsText() para convertirlo a texto WKT

query_geo = """
    (SELECT
        id,
        nombre,
        ciudad,
        ST_X(geom)      AS longitud,
        ST_Y(geom)      AS latitud,
        ST_AsText(geom) AS wkt_geometry
     FROM ubicaciones
     ORDER BY nombre) AS geo_query
"""

df_geo = spark.read.jdbc(url=JDBC_URL, table=query_geo, properties=jdbc_props)

print("Ubicaciones (con geometria como WKT):")
display(df_geo)

---
## Sección 6 — Escribir resultados de Spark a PostgreSQL

El flujo completo: procesamos datos en Spark y guardamos el resultado en PostgreSQL.

In [ ]:
# Calcular resumen por categoria con Spark
from pyspark.sql import functions as F

df_resumen = (
    df_spark
    .groupBy("categoria")
    .agg(
        F.count("*").alias("total_registros"),
        F.round(F.avg("valor"), 2).alias("promedio_valor"),
        F.max("valor").alias("valor_maximo"),
        F.min("valor").alias("valor_minimo")
    )
)

print("Resumen calculado con Spark:")
display(df_resumen)

In [ ]:
# Guardar el resumen en una nueva tabla de PostgreSQL
# mode="overwrite" crea la tabla si no existe, o la reemplaza
# mode="append" agrega filas a una tabla existente

(
    df_resumen
    .write
    .jdbc(
        url        = JDBC_URL,
        table      = "resumen_por_categoria",
        mode       = "overwrite",
        properties = jdbc_props
    )
)

print("Resumen guardado en la tabla 'resumen_por_categoria' de PostgreSQL.")

In [ ]:
# Verificar que la tabla fue creada correctamente
df_check = spark.read.jdbc(
    url=JDBC_URL, table="resumen_por_categoria", properties=jdbc_props
)
print("Tabla 'resumen_por_categoria' en PostgreSQL:")
display(df_check)

---
## Sección 7 — Ejercicio integrador

El siguiente pipeline ejecuta el flujo completo en un solo bloque:

```
Blob Storage  →  Spark (procesar)  →  PostgreSQL (guardar)
                      |                     |
                  (leer CSV)          (leer tabla geo)
```

In [ ]:
from pyspark.sql import functions as F

print("Iniciando pipeline Blob → Spark → PostgreSQL...")
print()

# Paso 1: leer CSV desde Blob
print("  [1/4] Leyendo CSV desde Blob Storage...")
df_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(abfs_path)
print(f"        {df_raw.count()} filas leidas.")

# Paso 2: leer ubicaciones desde PostgreSQL
print("  [2/4] Leyendo ubicaciones desde PostgreSQL...")
query_locs = "(SELECT id, nombre, ciudad, ST_X(geom) lon, ST_Y(geom) lat FROM ubicaciones) AS locs"
df_locs = spark.read.jdbc(url=JDBC_URL, table=query_locs, properties=jdbc_props)
print(f"        {df_locs.count()} ubicaciones leidas.")

# Paso 3: transformar datos con Spark
print("  [3/4] Procesando con Spark...")
df_procesado = (
    df_raw
    .withColumn("valor_normalizado", F.round(F.col("valor") / F.max("valor").over(
        __import__('pyspark.sql.window', fromlist=['Window']).Window.partitionBy("categoria")
    ), 4))
    .select("fecha", "nombre", "categoria", "valor", "valor_normalizado")
)

# Paso 4: guardar resultado en PostgreSQL
print("  [4/4] Guardando resultados en PostgreSQL...")
df_procesado.write.jdbc(
    url=JDBC_URL, table="datos_procesados", mode="overwrite", properties=jdbc_props
)
print()
print(f"Pipeline completado. Tabla 'datos_procesados' creada en PostgreSQL.")

display(df_procesado)

---

**Resumen del notebook de Databricks:**
- Configuraste ABFS para acceder a Blob Storage desde Spark
- Leiste un CSV de la nube sin descargarlo
- Ejecutaste consultas SQL sobre los datos con Spark SQL
- Conectaste a PostgreSQL via JDBC y leiste tablas (incluyendo geometrias)
- Escribiste resultados de Spark directamente en PostgreSQL

El Día 4 usaremos estos datos como entrada para servicios de ML en Azure.